## inference for proportion differences 

In [12]:
# # _rd_00.py 
# 비율 차이 추론. 평균차 분산, 표준오차, 신뢰구간, 가설검정  
#   등분산 전제, 이분산 전제. 오류 정정(자유도 관련 계산)

import os
import numpy as np                          # numpy 라이브러리 전체. 
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네. 
import scipy as sci 
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis 
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.weightstats import DescrStatsW, ztest
from scipy.stats import ttest_1samp, ttest_ind, f


In [13]:

# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat_r = pd.read_csv(dat_url) 
df_dat_r.head() 


,i,gender,ht
0,1,1,159.9
1,2,2,157.5
2,3,2,158.0
3,4,2,154.2
4,5,1,163.3


In [14]:
# # 1. 연습용 자료, 단순 무작위 추출 (2%, 대략 400개)
n_frac = 0.02 
n_ttl = len(df_dat_r)

n_smpld = int(n_ttl*n_frac)   
df_ss1 = df_dat_r.sample(n_smpld, replace=False, random_state=42)
# simple_sample = df_pop.sample(n=25, replace=False, random_state=42)
# 표본 갯수 n, 비복원추출(이게 디폴트), 검증위한 시드, 42 for fun. 
df_ss1.head()


,i,gender,ht
15561,15562,1,170.9
13056,13057,2,144.4
5702,5703,1,170.5
3062,3063,2,158.5
9199,9200,2,155.1


In [20]:
# # 2. 연습용 자료, 층화 표본 추출 (그룹별로 2%씩 비율 유지 추출) 
# Better Coding,... 
df_ss2 = (
    df_dat_r.groupby('gender', group_keys=False)
    .sample(frac=n_frac, random_state=42)
)
df_ss2.head()


,i,gender,ht
15862,15863,1,163.1
7185,7186,1,167.5
4804,4805,1,171.5
13485,13486,1,178.6
14926,14927,1,163.5


In [16]:
# df1의 속성들 알아내기.
# 전체 종합 요약	df.info()	행/열/타입/결측치 한눈에 보기
# 행/열 차원 크기	df.shape	(행 개수, 열 개수) 튜플 반환
# 자료 개수 (행)	len(df) 또는 df.shape[0]	데이터 건수
# 변수 개수 (열)	len(df.columns) 또는 df.shape[1]	컬럼 개수
# 변수 이름 목록	df.columns
df_ss1.info()
df_ss2.info()
# df1.shape[0]
# df1.shape[1]
# df1.columns
# print(df_ss1.shape, "\n",  df_ss1.columns) 
print(len(df_dat_r), len(df_ss1), len(df_ss2)) 


<class 'pandas.core.frame.DataFrame'>
Index: 402 entries, 15561 to 8046
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   i       402 non-null    int64  
 1   gender  402 non-null    int64  
 2   ht      402 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 12.6 KB
<class 'pandas.core.frame.DataFrame'>
Index: 403 entries, 15862 to 16411
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   i       403 non-null    int64  
 1   gender  403 non-null    int64  
 2   ht      403 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 12.6 KB
20128 402 403


In [ ]:
# df_ss1(단순추출 자료셋) 또는 df_ss2(층화추롤 자료셋) 을 분석함. 
# df_dat_r 은 그냥 놔두고.
# 이 단계에서 샘플 자료셋을 df_dat으로 명명하고, 분석. 
# 필요한 변수들 생성, 통합.

df_dat = df_ss1    # df_ss1을 다시 명명하고, 분석함. 

gn = df_dat['gender'].to_numpy()   # 이렇게 하면 배열로 전환. dataframe -> NumPy 배열
hgt = df_dat['ht'].to_numpy()
gndr = (gn == 1).astype(int)        # gn = 1, 2; gndr = 1, 0. 1/0으로 전환. 
df_dat['gndr'] = gndr               #   이것은 위에 1/0 자료를 추가. 성별임.

hgt_m = hgt[gndr == 1]  # 성별 1, m 그룹 키. 그룹별 자료 분리, 배열은 hgt 하나임.
hgt_f = hgt[gndr == 0]  #     0, f. 키 그룹별 자료 분리

n_all = len(df_dat_r)
n_now = len(df_dat) 
n_m = len(hgt_m)
n_f = len(hgt_f)

print(n_all, n_now, n_m, n_f)


20128 402 174 228


### 키를 기준으로 그룹을 나누자.
큰 키 그룹의 남자(성별=1) 비율, 작은 키 그룹의 남자 비율, 이 둘은 같은지 검토.

In [22]:
# 그룹을 평균키 기준으로 구분하는 아이디어. 연습.
hgt_ref = np.mean(hgt)          # 그룹 구분 참고값. 평균 이상, 이하.
hgt_grp = np.where(hgt > hgt_ref, "Tall", "Short")   # 기준 키 대비 Tall, Short.
df_dat['ht_g'] = hgt_grp            # data frame에 변수 추가된 것임. 
df_dat.head()


,i,gender,ht,gndr,ht_g
15561,15562,1,170.9,1,Tall
13056,13057,2,144.4,0,Short
5702,5703,1,170.5,1,Tall
3062,3063,2,158.5,0,Short
9199,9200,2,155.1,0,Short


In [ ]:

mf_tall = df_dat[df_dat['ht_g'] == 'Tall']['gndr']   # Tall 그룹 성별 'gndr' 을 받음, 1/0.  
mf_shrt = df_dat[df_dat['ht_g'] == 'Short']['gndr']  # Tall 그룹 성별 'gndr' 을 받음, 1/0.  

# dat.info()  # 5개 변수 
# Tall group 
n_tall = (df_dat['ht_g'] == 'Tall').sum()                              # Tall group, 관측치 수 
m_tall = ((df_dat['ht_g'] == 'Tall') & (df_dat['gndr'] == 1)).sum()    # Tall group, 남자(성=1) 수 

# Short group 
n_shrt = len(df_dat) - n_tall                                          # Short group, 관측치 수
m_shrt = ((df_dat['ht_g'] == 'Short') & (df_dat['gndr'] == 1)).sum()   # Short group, 남자(성=1) 수 

print("  m1   m2   n1   n2 ")
print(m_tall, m_shrt, n_tall, n_shrt) 

phat_m_tall = m_tall / n_tall   # Tall group, 남자 비율 
phat_m_shrt = m_shrt / n_shrt   # Short group, 남자 비율 

print("남 비율, Tall  group:  ", phat_m_tall ) 
print("남 비율, Short group:  ", phat_m_shrt ) 


  m1   m2   n1   n2 
154 20 192 210
남 비율, Tall  group:   0.8020833333333334
남 비율, Short group:   0.09523809523809523


\begin{align}
 H_0 : \Delta  = \Delta_0  \\
 H_A : \Delta \ne \Delta_0 
\end{align}

\begin{align}
\hat\Delta 
   & = \hat p_{tall} - \hat p_{short}  \\
   & \sim N(\Delta, SE^2 ) \\
SE^2 & = { p_{tall} (1-p_{tall}) \over n_{tall} }  + { p_{short} (1-p_{short}) \over n_{short} } \\  
   & =    p_{pool} (1-p_{pool}) \left( { 1 \over n_{tall} }   + { 1 \over n_{short} } \right) \\  
T_0 &= {\hat\Delta - \Delta_0 \over SE } \\
  & \sim N(0,1)
\end{align}

In [41]:

phat_diff = phat_m_tall - phat_m_shrt 
pool_p = (m_tall + m_shrt)/(n_tall + n_shrt) 

phat_dff_var = pool_p * ( 1- pool_p) *( 1 / n_tall +  1 / n_shrt )
phat_dff_se = phat_dff_var**0.5 

p_diff_zero = 0 
t_0 = ( phat_diff - p_diff_zero ) / phat_dff_se  
# pval = 2 * (1 - stats.norm.cdf( abs(t_0) ) )  # .cdf()와 .sf()에 미묘한 차이가 있네. 
pval = 2 * ( stats.norm.sf( abs(t_0) ) )        # 파이썬은 .sf()를 사용, 권장하는 듯.

print("H0: p_difference =", p_diff_zero) 
print("t statisticse:", t_0 )
print("  p-value :   ",  pval)


print("check with below and confirm calculation  \n ")

# do it with built-in modules
# m f ratio out of tall and short 
from statsmodels.stats.proportion import proportions_ztest
count = [m_tall, m_shrt] 
n_obs = [n_tall, n_shrt]   # check with number of obs.

z_stat, p_value = proportions_ztest(count, n_obs)
print(" 모듈로 반환한 분석 결과 ")
print("z-statistic:", z_stat)
print("p-value:", p_value)



H0: p_difference = 0
t statisticse: 14.287478082823409
  p-value :    2.619240133194895e-46
check with below and confirm calculation  
 
 모듈로 반환한 분석 결과 
z-statistic: 14.287478082823409
p-value: 2.619240133194895e-46


In [19]:
# 4. 코드 체크
#  변수 특성, 배열 형식, 계산 완료 등.
#    print('Saved figure to', saved_path)
print("Computing OK")


Computing OK
